In [23]:
from google.colab import drive
drive.mount('/content/drive')
#

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Model Training & Comparison

This notebook focuses on building, training, and evaluating multiple machine learning models for fraud detection using both the original imbalanced dataset and oversampled training data.

The objective is not only to identify the best-performing model, but also to understand how different model families behave under severe class imbalance conditions commonly found in fraud detection systems.

The notebook includes:

* baseline model development,
* linear and probabilistic models,
* tree-based models,
* boosted ensemble models,
* performance evaluation using fraud-focused metrics,
* and comparative analysis between original and oversampled training scenarios.

Special attention is placed on recall, precision, PR-AUC, and fraud detection tradeoffs rather than relying solely on accuracy, since fraud datasets are highly imbalanced and require more robust evaluation strategies.


In [24]:
%pip install catboost xgboost lightgbm

In [25]:
# data manipulation
import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# warning
import warnings
warnings.filterwarnings('ignore')

# baseline model
from sklearn.dummy import DummyClassifier

# linear model
from sklearn.linear_model import (LogisticRegression, SGDClassifier)

from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

# Tree models
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier)

# Boosted ensemble models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

import joblib


In [26]:
# load dataset
X_train_linear = joblib.load('/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/processed/X_train_linear.pkl')
X_train_linear_smote = joblib.load('/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/processed/X_train_linear_smote.pkl')
X_train_tree = joblib.load('/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/processed/X_train_tree.pkl')
X_train_tree_smote = joblib.load('/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/processed/X_train_tree_smote.pkl')

X_test_linear = joblib.load('/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/processed/X_test_linear.pkl')
X_test_tree = joblib.load('/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/processed/X_test_tree.pkl')

y_train = joblib.load('/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/processed/y_train.pkl')
y_train_linear_smote = joblib.load('/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/processed/y_train_linear_smote.pkl')
y_train_tree_smote = joblib.load('/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/processed/y_train_tree_smote.pkl')

y_test = joblib.load('/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/processed/y_test.pkl')

#

In [27]:
# =========================
# FEATURES TO REMOVE
# =========================
drop_features = ["origin_balance_error", "destination_balance_error"]


# =========================
# LINEAR DATASETS
# =========================
X_train_linear = X_train_linear.drop(columns=drop_features)

X_test_linear = X_test_linear.drop(columns=drop_features)

X_train_linear_smote = X_train_linear_smote.drop(columns=drop_features)


# =========================
# TREE DATASETS
# =========================
X_train_tree = X_train_tree.drop(columns=drop_features)

X_test_tree = X_test_tree.drop(columns=drop_features)

X_train_tree_smote = X_train_tree_smote.drop(columns=drop_features)

In [28]:
# check dataset shape
print(f'X_train_linear shape: {X_train_linear.shape}')
print(f'\nX_train_linear_smote shape: {X_train_linear_smote.shape}')
print(f'\nX_train_tree shape: {X_train_tree.shape}')
print(f'\nX_train_tree_smote shape: {X_train_tree_smote.shape}')
print(f'\nX_test_linear shape: {X_test_linear.shape}')
print(f'\nX_test_tree shape: {X_test_tree.shape}')
print(f'\ny_train shape: {y_train.shape}')
print(f'\ny_train_linear_smote shape: {y_train_linear_smote.shape}')
print(f'\ny_train_tree_smote shape: {y_train_tree_smote.shape}')
print(f'\ny_test shape: {y_test.shape}')

#

X_train_linear shape: (800000, 13)

X_train_linear_smote shape: (958760, 13)

X_train_tree shape: (800000, 13)

X_train_tree_smote shape: (958760, 13)

X_test_linear shape: (200000, 13)

X_test_tree shape: (200000, 13)

y_train shape: (800000,)

y_train_linear_smote shape: (958760,)

y_train_tree_smote shape: (958760,)

y_test shape: (200000,)


In [29]:
# =========================
# ORIGINAL TRAINING DISTRIBUTION
# =========================
print("Original Training Distribution:")
print(y_train.value_counts(normalize=True))


# =========================
# SMOTE DISTRIBUTION
# =========================
print("\nSMOTE Training Distribution:")
print(y_train_linear_smote.value_counts(normalize=True))


# =========================
# TEST DISTRIBUTION
# =========================
print("\nTest Distribution:")
print(y_test.value_counts(normalize=True))

Original Training Distribution:
isFraud
0    0.998709
1    0.001291
Name: proportion, dtype: float64

SMOTE Training Distribution:
isFraud
0    0.833334
1    0.166666
Name: proportion, dtype: float64

Test Distribution:
isFraud
0    0.99871
1    0.00129
Name: proportion, dtype: float64


In [30]:
# store all model evaluation results
results = []

In [31]:
# Build Reusable Evaluation Function
def evaluate_model(model, model_name, family, training_type, X_train, y_train, X_test, y_test):

  # loop through models
  for model_name, model in model.items():

    print(f"\nTraining {model_name} ({training_type})...\n")


    # =========================
    # Train model
    # =========================
    model.fit(X_train, y_train)

    # =========================
    # Prediction
    # =========================
    y_pred = model.predict(X_test)

    # =========================
    # Probability prediction
    # =========================
    y_proba = model.predict_proba(X_test)[:, 1]

    # =========================
    # Evaluation
    # =========================
    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(y_test, y_pred)

    recall = recall_score(y_test, y_pred)

    f1 = f1_score(y_test, y_pred)

    roc_auc = roc_auc_score(y_test, y_proba)

    # ============================
    # store result
    # ============================
    results.append({
        'model': model_name,
        'family': family,
        'training_type': training_type,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': roc_auc
    })

    # =========================
      # PRINT SUMMARY
      # =========================
    print(f"\n{model_name} ({training_type}) Completed\n")

    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")

## **Train DummyClassifier**

In [32]:
# =========================
# DUMMY CLASSIFIER
# =========================
dummy_model = {
    'dummy': DummyClassifier(strategy='most_frequent')
}

# evaluate dummy model
evaluate_model(dummy_model, 'dummy', 'baseline', 'original', X_train_tree, y_train, X_test_tree, y_test)


Training dummy (original)...


dummy (original) Completed

Accuracy:  0.9987
Precision: 0.0000
Recall:    0.0000
F1 Score:  0.0000
ROC-AUC:   0.5000


In [33]:
# convert results into dataframe
results_df = pd.DataFrame(results)

# preview
results_df

,model,family,training_type,accuracy,precision,recall,f1_score,roc_auc
0,dummy,baseline,original,0.99871,0.0,0.0,0.0,0.5


## Dummy Classifier Baseline

The Dummy Classifier was used as the baseline model before training the actual machine learning models. The model predicted all transactions as non-fraud because the dataset is highly imbalanced.

Although the model achieved a very high accuracy score, it failed completely at detecting fraudulent transactions, resulting in zero precision, recall, and F1-score. The ROC-AUC score of 0.50 also showed that the model had no real learning capability.

This confirms that accuracy is not a reliable metric for fraud detection problems with severe class imbalance. A model can achieve high accuracy simply by predicting the majority class while missing all fraud cases.

The baseline result provides a reference point that subsequent machine learning models must outperform.


## **Linear & Distance-Based models**

In [34]:
# =========================
# LINEAR & DISTANCE-BASED MODELS
# =========================

linear_model = {
    'logistic_regression': LogisticRegression(random_state=42, max_iter=1000),
    'sgd_classifier': SGDClassifier(loss="log_loss",random_state=42),
    'Gaussian naive_bayes': GaussianNB(),
    'knn': KNeighborsClassifier(n_neighbors=5)
}

# train and evaluate models
evaluate_model(linear_model, 'linear_model',
               'Linear & Distance-Based',
               'original', X_train_linear, y_train,
               X_test_linear, y_test)


Training logistic_regression (original)...


logistic_regression (original) Completed

Accuracy:  0.9995
Precision: 0.9251
Recall:    0.6705
F1 Score:  0.7775
ROC-AUC:   0.9894

Training sgd_classifier (original)...


sgd_classifier (original) Completed

Accuracy:  0.9992
Precision: 0.8452
Recall:    0.5078
F1 Score:  0.6344
ROC-AUC:   0.9448

Training Gaussian naive_bayes (original)...


Gaussian naive_bayes (original) Completed

Accuracy:  0.8392
Precision: 0.0077
Recall:    0.9612
F1 Score:  0.0152
ROC-AUC:   0.9766

Training knn (original)...


knn (original) Completed

Accuracy:  0.9994
Precision: 1.0000
Recall:    0.5659
F1 Score:  0.7228
ROC-AUC:   0.8700


In [35]:
results_df = pd.DataFrame(results)

# preview
results_df

,model,family,training_type,accuracy,precision,recall,f1_score,roc_auc
0,dummy,baseline,original,0.998710,0.000000,0.000000,0.000000,0.500000
1,logistic_regression,Linear & Distance-Based,original,0.999505,0.925134,0.670543,0.777528,0.989357
2,sgd_classifier,Linear & Distance-Based,original,0.999245,0.845161,0.507752,0.634383,0.944781
3,Gaussian naive_bayes,Linear & Distance-Based,original,0.839180,0.007654,0.961240,0.015187,0.976577
4,knn,Linear & Distance-Based,original,0.999440,1.000000,0.565891,0.722772,0.870045


## Linear & Distance-Based Models — Original Data

The models were trained on the original imbalanced dataset to evaluate their performance under realistic fraud conditions.

Logistic Regression gave the best overall performance in this group, with strong precision, recall, and ROC-AUC scores. It was able to detect fraud effectively while keeping false alarms relatively low.

SGDClassifier performed reasonably well but missed more fraud cases compared to Logistic Regression.

Gaussian Naive Bayes achieved very high recall but very low precision, meaning it detected many fraud cases but also incorrectly flagged many legitimate transactions as fraud.

KNN showed very high precision but lower recall, indicating that it was conservative in predicting fraud and missed several fraudulent transactions.

Overall, the results show the tradeoff between detecting more fraud cases and reducing false positives.


## **Train Same Models on SMOTE Data**

In [36]:
evaluate_model(linear_model, 'linear_model',
               'Linear & Distance-Based', 'SMOTE',
               X_train_linear_smote, y_train_linear_smote,
               X_test_linear, y_test)
#


Training logistic_regression (SMOTE)...


logistic_regression (SMOTE) Completed

Accuracy:  0.9952
Precision: 0.2029
Recall:    0.9341
F1 Score:  0.3333
ROC-AUC:   0.9862

Training sgd_classifier (SMOTE)...


sgd_classifier (SMOTE) Completed

Accuracy:  0.9898
Precision: 0.0943
Recall:    0.8062
F1 Score:  0.1689
ROC-AUC:   0.9716

Training Gaussian naive_bayes (SMOTE)...


Gaussian naive_bayes (SMOTE) Completed

Accuracy:  0.8091
Precision: 0.0066
Recall:    0.9806
F1 Score:  0.0131
ROC-AUC:   0.9777

Training knn (SMOTE)...


knn (SMOTE) Completed

Accuracy:  0.9985
Precision: 0.4482
Recall:    0.8217
F1 Score:  0.5800
ROC-AUC:   0.9201


In [37]:
results_df = pd.DataFrame(results)

# preview
results_df


,model,family,training_type,accuracy,precision,recall,f1_score,roc_auc
0,dummy,baseline,original,0.998710,0.000000,0.000000,0.000000,0.500000
1,logistic_regression,Linear & Distance-Based,original,0.999505,0.925134,0.670543,0.777528,0.989357
2,sgd_classifier,Linear & Distance-Based,original,0.999245,0.845161,0.507752,0.634383,0.944781
3,Gaussian naive_bayes,Linear & Distance-Based,original,0.839180,0.007654,0.961240,0.015187,0.976577
4,knn,Linear & Distance-Based,original,0.999440,1.000000,0.565891,0.722772,0.870045
5,logistic_regression,Linear & Distance-Based,SMOTE,0.995180,0.202862,0.934109,0.333333,0.986228
6,sgd_classifier,Linear & Distance-Based,SMOTE,0.989765,0.094331,0.806202,0.168900,0.971604
7,Gaussian naive_bayes,Linear & Distance-Based,SMOTE,0.809100,0.006584,0.980620,0.013080,0.977698
8,knn,Linear & Distance-Based,SMOTE,0.998465,0.448203,0.821705,0.580027,0.920139


### Effect of SMOTE on Model Performance

The SMOTE-resampled dataset improved recall across most models, meaning the models became better at detecting fraudulent transactions. However, this improvement also introduced a noticeable drop in precision, showing that many legitimate transactions were incorrectly classified as fraud.

For Logistic Regression, the original imbalanced dataset produced a more balanced performance with very high precision and moderate recall, while the SMOTE version became much more aggressive in fraud detection but generated significantly more false positives.

A similar pattern was observed for SGDClassifier and KNN, where oversampling increased fraud sensitivity but reduced prediction reliability.

These results suggest that although SMOTE helps models learn minority fraud patterns better, it may also cause the models to overpredict fraud within this dataset. This highlights the importance of balancing fraud detection capability with false positive control in real-world financial systems.


## **Tree-Based & Ensemble Models**

In [38]:
# initialize models
tree_model = {
    'decision_tree': DecisionTreeClassifier(random_state=42),

    'random_forest': RandomForestClassifier(random_state=42, n_estimators=100, n_jobs=-1),

    'gradient_boosting': GradientBoostingClassifier(random_state=42, n_estimators=100),

    'adaboost': AdaBoostClassifier(random_state=42, n_estimators=100),

    'xgboost': XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1),

    'lightgbm': LGBMClassifier(n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        verbose=0,
        random_state=42,
        n_jobs=-1
    ),

    'catboost': CatBoostClassifier(
        iterations=100,
        learning_rate=0.1,
        depth=6,
        verbose=0,
        random_state=42
    )

}

In [39]:
# train model
evaluate_model(tree_model, 'tree_model',
               'Tree-Based & Ensemble',
               'original', X_train_tree, y_train,
               X_test_tree, y_test)
#


Training decision_tree (original)...


decision_tree (original) Completed

Accuracy:  0.9995
Precision: 0.8481
Recall:    0.7791
F1 Score:  0.8121
ROC-AUC:   0.8894

Training random_forest (original)...


random_forest (original) Completed

Accuracy:  0.9997
Precision: 0.9847
Recall:    0.7481
F1 Score:  0.8502
ROC-AUC:   0.9920

Training gradient_boosting (original)...


gradient_boosting (original) Completed

Accuracy:  0.9987
Precision: 0.5122
Recall:    0.2442
F1 Score:  0.3307
ROC-AUC:   0.5789

Training adaboost (original)...


adaboost (original) Completed

Accuracy:  0.9994
Precision: 0.9554
Recall:    0.5814
F1 Score:  0.7229
ROC-AUC:   0.9935

Training xgboost (original)...


xgboost (original) Completed

Accuracy:  0.9997
Precision: 0.9395
Recall:    0.7829
F1 Score:  0.8541
ROC-AUC:   0.9978

Training lightgbm (original)...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


In [40]:
results_df = pd.DataFrame(results)

# preview
results_df

,model,family,training_type,accuracy,precision,recall,f1_score,roc_auc
0,dummy,baseline,original,0.998710,0.000000,0.000000,0.000000,0.500000
1,logistic_regression,Linear & Distance-Based,original,0.999505,0.925134,0.670543,0.777528,0.989357
2,sgd_classifier,Linear & Distance-Based,original,0.999245,0.845161,0.507752,0.634383,0.944781
3,Gaussian naive_bayes,Linear & Distance-Based,original,0.839180,0.007654,0.961240,0.015187,0.976577
4,knn,Linear & Distance-Based,original,0.999440,1.000000,0.565891,0.722772,0.870045
5,logistic_regression,Linear & Distance-Based,SMOTE,0.995180,0.202862,0.934109,0.333333,0.986228
6,sgd_classifier,Linear & Distance-Based,SMOTE,0.989765,0.094331,0.806202,0.168900,0.971604
7,Gaussian naive_bayes,Linear & Distance-Based,SMOTE,0.809100,0.006584,0.980620,0.013080,0.977698
8,knn,Linear & Distance-Based,SMOTE,0.998465,0.448203,0.821705,0.580027,0.920139
9,decision_tree,Tree-Based & Ensemble,original,0.999535,0.848101,0.779070,0.812121,0.889445


## **Tree-Based & Ensemble Models SMOTE**

In [ ]:
evaluate_model(tree_model, 'tree_model',
               'Tree-Based & Ensemble',
               'SMOTE', X_train_tree_smote, y_train_tree_smote,
               X_test_tree, y_test)
#


Training decision_tree (SMOTE)...


decision_tree (SMOTE) Completed

Accuracy:  0.9991
Precision: 0.5915
Recall:    0.8643
F1 Score:  0.7024
ROC-AUC:   0.9318

Training random_forest (SMOTE)...


random_forest (SMOTE) Completed

Accuracy:  0.9993
Precision: 0.6522
Recall:    0.9302
F1 Score:  0.7668
ROC-AUC:   0.9959

Training gradient_boosting (SMOTE)...


gradient_boosting (SMOTE) Completed

Accuracy:  0.9958
Precision: 0.2290
Recall:    0.9496
F1 Score:  0.3690
ROC-AUC:   0.9924

Training adaboost (SMOTE)...


adaboost (SMOTE) Completed

Accuracy:  0.9874
Precision: 0.0872
Recall:    0.9302
F1 Score:  0.1595
ROC-AUC:   0.9900

Training xgboost (SMOTE)...


xgboost (SMOTE) Completed

Accuracy:  0.9974
Precision: 0.3264
Recall:    0.9767
F1 Score:  0.4893
ROC-AUC:   0.9967

Training lightgbm (SMOTE)...


lightgbm (SMOTE) Completed

Accuracy:  0.9974
Precision: 0.3320
Recall:    0.9767
F1 Score:  0.4956
ROC-AUC:   0.9947

Training catboost (SMOTE)...


catboost (SMOTE) Completed

Accu

In [ ]:
results_df = pd.DataFrame(results)

# preview
results_df

,model,family,training_type,accuracy,precision,recall,f1_score,roc_auc
0,dummy,baseline,original,0.998710,0.000000,0.000000,0.000000,0.500000
1,logistic_regression,Linear & Distance-Based,original,0.999505,0.925134,0.670543,0.777528,0.989357
2,sgd_classifier,Linear & Distance-Based,original,0.999245,0.845161,0.507752,0.634383,0.944781
3,Gaussian naive_bayes,Linear & Distance-Based,original,0.839180,0.007654,0.961240,0.015187,0.976577
4,knn,Linear & Distance-Based,original,0.999440,1.000000,0.565891,0.722772,0.870045
5,logistic_regression,Linear & Distance-Based,SMOTE,0.995180,0.202862,0.934109,0.333333,0.986228
6,sgd_classifier,Linear & Distance-Based,SMOTE,0.989765,0.094331,0.806202,0.168900,0.971604
7,Gaussian naive_bayes,Linear & Distance-Based,SMOTE,0.809100,0.006584,0.980620,0.013080,0.977698
8,knn,Linear & Distance-Based,SMOTE,0.998465,0.448203,0.821705,0.580027,0.920139
9,decision_tree,Tree-Based & Ensemble,original,0.999535,0.848101,0.779070,0.812121,0.889445


### Effect of SMOTE on Tree-Based & Ensemble Models

The SMOTE-resampled dataset increased recall across almost all tree-based and ensemble models, showing that the models became significantly more sensitive to fraudulent transactions. However, this improvement came with a noticeable reduction in precision, indicating a larger number of false positive predictions.

For Random Forest, recall improved substantially after SMOTE, but precision dropped compared to the original imbalanced training setup. A similar pattern was observed across XGBoost, LightGBM, CatBoost, AdaBoost, and Decision Tree models, where fraud detection sensitivity increased while prediction reliability decreased.

Interestingly, XGBoost and Random Forest trained on the original imbalanced dataset still produced the strongest overall balance between precision, recall, F1-score, and ROC-AUC. This suggests that advanced ensemble methods were already capable of learning meaningful fraud patterns without relying heavily on synthetic oversampling.

Gradient Boosting showed improved recall under SMOTE but continued to underperform compared to the stronger ensemble models, particularly in precision and F1-score.

Overall, the results indicate that although SMOTE improves fraud detection coverage, advanced tree-based ensembles achieved more stable and balanced performance when trained on the original imbalanced dataset.


In [ ]:
# save results dataframe
results_df.to_csv('/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/processed/results.csv', index=False)